# Crawl Corpus từ Q&A Aspects

Pipeline:
1. Load **viendinhduong_qa.jsonl** + **eval_split_1..5.jsonl** (deduplicated)
2. Dùng Groq Qwen3 để tách mỗi câu trả lời thành các **aspects**
3. Với mỗi aspect → **DuckDuckGo search** (loại trừ domain nguồn)
4. **Crawl** bài tìm được → score bằng **cross-encoder**
5. Lưu vào **corpus format** (chunk_id, doc_id, source, url, ...)

**Resume-safe**: Nếu Colab reset, chạy lại từ đầu — script sẽ bỏ qua các câu đã xong.

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Cài dependencies

In [ ]:
!pip install -q groq ddgs trafilatura sentence-transformers python-dotenv

## 2. Config — chỉnh ở đây

In [ ]:
import os

# ── API Key ───────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print("Lấy API key từ Colab Secrets OK")
except Exception:
    GROQ_API_KEY = "gsk_xxx"   # <-- thay bằng key thật
    print("Dùng API key nhập trực tiếp")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# ── Paths ─────────────────────────────────────────────────────────────
DRIVE = "/content/drive/MyDrive/Callbot"   # <-- thay nếu folder Drive khác

INPUT_FILES = [
    f"{DRIVE}/evaluation/eval_split_1.jsonl",
    f"{DRIVE}/evaluation/eval_split_2.jsonl",
    f"{DRIVE}/evaluation/eval_split_3.jsonl",
    f"{DRIVE}/evaluation/eval_split_4.jsonl",
    f"{DRIVE}/evaluation/eval_split_5.jsonl",
]

OUTPUT_CORPUS  = f"{DRIVE}/data_final/corpus_from_aspects.jsonl"
DONE_IDS_FILE  = f"{DRIVE}/data_final/corpus_from_aspects_done.txt"
FAILED_LOG     = f"{DRIVE}/data_final/corpus_from_aspects_failed.txt"

# ── Model & pipeline params ───────────────────────────────────────────
GROQ_MODEL         = "qwen/qwen3-32b"
RERANK_MODEL       = "BAAI/bge-reranker-v2-m3"
RERANK_THRESHOLD   = 0.0

MAX_SEARCH_RESULTS = 8
MAX_URLS_PER_QA    = 20
TOP_ARTICLES       = 3
ASPECTS_PER_QA     = 5

SEARCH_PAUSE = 2.5
CRAWL_PAUSE  = 0.5

SOURCE_DOMAIN_MAP = {
    "viendinhduong":  "viendinhduong.vn",
    "benhvienthucuc": "benhvienthucuc.vn",
}

os.makedirs(f"{DRIVE}/data_final", exist_ok=True)
print("Config OK")

## 3. Load & merge input files

In [ ]:
import json
import hashlib
from pathlib import Path
from urllib.parse import urlparse

def normalize_sample(raw: dict) -> dict | None:
    # Chỉ lấy câu thuộc viendinhduong
    if raw.get("source") != "viendinhduong":
        return None

    qa_id = str(raw.get("id") or raw.get("code") or raw.get("no") or "")
    src_domain = SOURCE_DOMAIN_MAP.get("viendinhduong", "viendinhduong.vn")

    return {
        "qa_id":      qa_id,
        "question":   raw.get("question", "").strip(),
        "answer":     raw.get("answer", "").strip(),
        "src_domain": src_domain,
    }


all_samples = []
seen_ids    = set()

for fpath in INPUT_FILES:
    p = Path(fpath)
    if not p.exists():
        print(f"  [SKIP] Không tìm thấy: {fpath}")
        continue
    before = len(all_samples)
    with open(p, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            raw = json.loads(line)
            s = normalize_sample(raw)
            if not s or not s["question"] or not s["answer"]:
                continue
            if not s["qa_id"]:
                s["qa_id"] = "q_" + hashlib.md5(s["question"].encode()).hexdigest()[:8]
            if s["qa_id"] in seen_ids:
                continue
            seen_ids.add(s["qa_id"])
            all_samples.append(s)
    print(f"  {p.name}: +{len(all_samples) - before} câu viendinhduong")

print(f"\nTổng: {len(all_samples)} câu")

## 4. Khởi tạo models

In [ ]:
from groq import Groq
from sentence_transformers import CrossEncoder

groq_client = Groq(api_key=GROQ_API_KEY)

print(f"Load reranker: {RERANK_MODEL} ...")
reranker = CrossEncoder(RERANK_MODEL)
print("Reranker ready.")

## 5. Helper functions

In [ ]:
import re
import time
import hashlib
import uuid
from urllib.parse import urlparse

import trafilatura
from ddgs import DDGS

# ── Aspect extraction ─────────────────────────────────────────────────

ASPECT_PROMPT = """\
Bạn là chuyên gia dinh dưỡng. Từ câu hỏi và câu trả lời dưới đây, hãy xác định \
các KHÍA CẠNH KIẾN THỨC CỤ THỂ cần tìm kiếm để có thể trả lời câu hỏi này.

Yêu cầu mỗi khía cạnh:
- Là một sự kiện/khái niệm độc lập, có thể tìm kiếm được trên internet
- Không trùng lặp nhau
- Tối đa {max_aspects} khía cạnh

Câu hỏi: {question}

Câu trả lời tham khảo: {answer}

Trả về JSON (không giải thích thêm):
{{
  "aspects": [
    {{
      "aspect": "mô tả ngắn khía cạnh",
      "query_vi": "từ khóa tìm kiếm tiếng Việt",
      "query_en": "english search query"
    }}
  ]
}}
"""


def extract_aspects(question: str, answer: str) -> list[dict]:
    prompt = ASPECT_PROMPT.format(
        question=question,
        answer=answer[:2000],
        max_aspects=ASPECTS_PER_QA,
    )
    try:
        resp = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=800,
            temperature=0.2,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
        raw = re.sub(r"```(?:json)?\s*", "", raw).replace("```", "").strip()
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if not match:
            return []
        data = json.loads(match.group())
        return data.get("aspects", [])
    except Exception as e:
        print(f"  [aspect extraction error] {e}")
        return []


# ── DuckDuckGo search ─────────────────────────────────────────────────

def ddg_search(query: str, exclude_domain: str, num: int = 8) -> list[str]:
    full_query = f"{query} -site:{exclude_domain}"
    try:
        with DDGS() as ddgs:
            results = ddgs.text(full_query, max_results=num)
            return [r.get("href", "") for r in results if r.get("href")]
    except Exception as e:
        print(f"  [ddg search error] {e}")
        return []


# ── Crawl & chunk ─────────────────────────────────────────────────────

def crawl_url(url: str) -> str | None:
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return None
        text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=False,
            no_fallback=False,
        )
        return text if text and len(text) > 200 else None
    except Exception:
        return None


def chunk_text(text: str, window: int = 200, stride: int = 150) -> list[str]:
    words = text.split()
    chunks = []
    for i in range(0, len(words), stride):
        chunk = " ".join(words[i: i + window])
        if len(chunk) >= 80:
            chunks.append(chunk)
    return chunks


def score_chunks(chunks: list[str], question: str, answer: str = "") -> list[dict]:
    """Score chunks dựa trên câu hỏi + câu trả lời (answer cung cấp thêm ngữ cảnh về thông tin cần tìm)."""
    if not chunks:
        return []
    # Kết hợp question + answer để score chính xác hơn — ưu tiên chunk chứa facts cụ thể
    query = question + (" " + answer[:300] if answer else "")
    pairs = [[query, c] for c in chunks]
    scores = reranker.predict(pairs, batch_size=32)
    return [
        {"text": c, "score": round(float(s), 4)}
        for c, s in zip(chunks, scores)
        if s >= RERANK_THRESHOLD
    ]


# ── Corpus format ─────────────────────────────────────────────────────

def make_chunk_id(seed: str) -> str:
    h = hashlib.md5(seed.encode()).hexdigest()
    return str(uuid.UUID(h))


def make_corpus_rows(url: str, text: str, source_domain: str, question: str) -> list[dict]:
    chunks = chunk_text(text)
    rows = []
    for i, chunk in enumerate(chunks):
        rows.append({
            "chunk_id":      make_chunk_id(f"{url}_{i}"),
            "doc_id":        url,
            "source":        source_domain,
            "url":           url,
            "title":         "",
            "category":      "dinh-duong",
            "chunk_index":   i,
            "text":          chunk,
            "embed_text":    chunk,
            "from_question": question,
        })
    return rows


print("Helper functions OK")

## 6. Chạy pipeline

In [ ]:
# ── Load done IDs (resume) ────────────────────────────────────────────
done_ids = set()
if Path(DONE_IDS_FILE).exists():
    with open(DONE_IDS_FILE, encoding="utf-8") as f:
        done_ids = {l.strip() for l in f if l.strip()}
    print(f"Resume: {len(done_ids)} câu đã xong, bỏ qua.")

pending = [s for s in all_samples if s["qa_id"] not in done_ids]
print(f"Còn lại: {len(pending)} câu cần xử lý")
print("=" * 60)

# ── Mở output files ───────────────────────────────────────────────────
fout    = open(OUTPUT_CORPUS,  "a", encoding="utf-8")
fdone   = open(DONE_IDS_FILE,  "a", encoding="utf-8")
ffailed = open(FAILED_LOG,     "a", encoding="utf-8")

total_chunks  = 0
total_success = 0

for idx, sample in enumerate(pending):
    qa_id      = sample["qa_id"]
    question   = sample["question"]
    answer     = sample["answer"]
    src_domain = sample["src_domain"]

    print(f"\n[{idx+1}/{len(pending)}] {qa_id}")
    print(f"  Q: {question[:80]}...")

    # Bước 1: Extract aspects
    aspects = extract_aspects(question, answer)
    if not aspects:
        print("  Không tách được aspects → fallback dùng câu hỏi gốc")
        aspects = []
    else:
        for a in aspects:
            print(f"    - {a['aspect']}")

    # Bước 2: Build queries
    seen_urls = set()
    candidate_urls = []

    all_queries = [question] + [
        q
        for aspect in aspects
        for q in [aspect.get("query_vi", ""), aspect.get("query_en", "")]
        if q
    ]

    for query in all_queries:
        urls = ddg_search(query, src_domain, num=MAX_SEARCH_RESULTS)
        for u in urls:
            if u not in seen_urls:
                seen_urls.add(u)
                candidate_urls.append(u)
        time.sleep(SEARCH_PAUSE)
        if len(candidate_urls) >= MAX_URLS_PER_QA:
            break

    candidate_urls = candidate_urls[:MAX_URLS_PER_QA]
    print(f"  URLs tìm được: {len(candidate_urls)}")

    if not candidate_urls:
        print("  Không tìm được URL")
        ffailed.write(f"{qa_id}\n"); ffailed.flush()
        continue

    # Bước 3: Crawl + score (dùng cả question + answer để ưu tiên chunk chứa facts cụ thể)
    scored_articles = []
    for url in candidate_urls:
        text = crawl_url(url)
        if not text:
            continue
        chunks = chunk_text(text)
        scored = score_chunks(chunks, question, answer)
        if scored:
            best = max(c["score"] for c in scored)
            scored_articles.append({"url": url, "text": text, "best_score": best})
        time.sleep(CRAWL_PAUSE)

    scored_articles.sort(key=lambda x: -x["best_score"])
    top = scored_articles[:TOP_ARTICLES]

    if not top:
        print("  Không tìm được bài liên quan")
        ffailed.write(f"{qa_id}\n"); ffailed.flush()
        continue

    # Bước 4: Lưu corpus
    n_chunks = 0
    for article in top:
        rows = make_corpus_rows(
            url=article["url"],
            text=article["text"],
            source_domain=urlparse(article["url"]).netloc,
            question=question,
        )
        for row in rows:
            fout.write(json.dumps(row, ensure_ascii=False) + "\n")
        n_chunks += len(rows)

    fout.flush()
    fdone.write(f"{qa_id}\n"); fdone.flush()

    total_chunks  += n_chunks
    total_success += 1
    print(f"  OK: {len(top)} bài, {n_chunks} chunks (best={top[0]['best_score']:.3f})")

fout.close(); fdone.close(); ffailed.close()

print(f"""
{'='*60}
XONG!
  Câu thành công : {total_success} / {len(pending)}
  Chunks mới     : {total_chunks}
  Output         : {OUTPUT_CORPUS}
  Done IDs       : {DONE_IDS_FILE}
  Failed log     : {FAILED_LOG}
{'='*60}
""")

## 7. Kiểm tra output

In [ ]:
import json
from pathlib import Path
from collections import Counter

rows = []
with open(OUTPUT_CORPUS, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

print(f"Tổng chunks: {len(rows)}")
print(f"Unique docs : {len(set(r['doc_id'] for r in rows))}")
print(f"Unique Q    : {len(set(r['from_question'] for r in rows))}")

# Top domains
domains = Counter(r['source'] for r in rows)
print("\nTop domains:")
for d, c in domains.most_common(10):
    print(f"  {d:<35} {c:>5} chunks")

# Preview
print("\nSample chunk:")
print(json.dumps(rows[0], ensure_ascii=False, indent=2))